### JDE Moving Average vs Prophet

**Objective:** Implement JDE-style Moving Average forecasting on the same dataset and validate against real January 2026 actuals — providing a rigorous baseline comparison against Prophet.

**Why this matters:**
JDE's native forecasting module uses Simple Moving Average (SMA) as its default method — no ML, no changepoint detection, no seasonality handling. By comparing Prophet against SMA on the **exact same holdout validation set**, we quantify the real-world improvement our ML solution delivers.

**Moving Average windows tested:** 3-month, 6-month, 12-month  
**Validation:** Real January 2026 F42119 actuals (same as Prophet validation)  
**Prophet benchmark:** Qty MAPE = 7.34%, Rev MAPE = 10.58%, Avg = 8.96%

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded successfully')

In [ ]:
monthly = pd.read_csv('data/processed/monthly_aggregated.csv', parse_dates=['ds'])
monthly = monthly.sort_values('ds').reset_index(drop=True)

print(f'Monthly data : {len(monthly)} months')
print(f'Date range   : {monthly["ds"].min().date()} to {monthly["ds"].max().date()}')
print()
monthly[['ds', 'total_qty', 'total_revenue']].tail(6)

In [ ]:
df_jan = pd.read_csv('jan_raw_data.csv', low_memory=False)
df_jan.columns = df_jan.columns.str.strip()

COLS_MAP = {
    'G/L Date'         : 'gl_date',
    'Quantity Shipped'  : 'qty_shipped',
    'Extended Price'    : 'extended_price',
    'Ln Ty'            : 'line_type'
}
df_jan = df_jan[[c for c in COLS_MAP if c in df_jan.columns]].rename(columns=COLS_MAP)
df_jan['qty_shipped']    = pd.to_numeric(df_jan['qty_shipped'].astype(str).str.replace(',',''), errors='coerce')
df_jan['extended_price'] = pd.to_numeric(df_jan['extended_price'].astype(str).str.replace(',',''), errors='coerce')
df_jan['gl_date']        = pd.to_datetime(df_jan['gl_date'], errors='coerce')
df_jan = df_jan[
    df_jan['extended_price'].notna() &
    (df_jan['line_type'].str.strip() == 'S') &
    (df_jan['qty_shipped'] > 0) &
    (df_jan['extended_price'] > 0) &
    df_jan['gl_date'].notna()
]

actual_jan26_qty = df_jan['qty_shipped'].sum()
actual_jan26_rev = df_jan['extended_price'].sum()

print('=== ACTUAL JANUARY 2026 ===')
print(f'Quantity : {actual_jan26_qty:,.0f} units')
print(f'Revenue  : LAK {actual_jan26_rev:,.0f}')
print(f'Revenue  : LAK {actual_jan26_rev/1e9:.2f} Billion')

### Implement Moving Average Methods

We implement 4 moving average variants that mirror JDE's native forecasting options:

| Method | Description | JDE Equivalent |
|---|---|---|
| SMA-3 | Simple 3-month average | JDE Balance Forward Method 1 |
| SMA-6 | Simple 6-month average | JDE Balance Forward Method 2 |
| SMA-12 | Simple 12-month average | JDE Percent Over Last Year |
| WMA-6 | Weighted 6-month (recent months weighted higher) | JDE Weighted Moving Average |

In [ ]:
def compute_mape(actual, predicted):
    """Compute MAPE for a single forecast point."""
    return abs((predicted - actual) / actual) * 100


def simple_moving_average(series, window):
    """
    Simple Moving Average — JDE default forecasting method.
    Uses the mean of the last N months to predict the next month.
    This is exactly what JDE Balance Forward method does.
    """
    return series.iloc[-window:].mean()


def weighted_moving_average(series, window):
    """
    Weighted Moving Average — more recent months get higher weight.
    JDE weighted moving average method.
    Weights: oldest month=1, newest month=N
    """
    data   = series.iloc[-window:].values
    weights = np.arange(1, window + 1)  # 1,2,3...N
    return np.average(data, weights=weights)


def exponential_smoothing(series, alpha=0.3):
    """
    Exponential Smoothing — gives exponentially decreasing
    weight to older observations.
    alpha=0.3 is a common default (similar to JDE smoothing constant).
    """
    values = series.values
    smoothed = [values[0]]
    for i in range(1, len(values)):
        smoothed.append(alpha * values[i] + (1 - alpha) * smoothed[-1])
    return smoothed[-1]


print('Moving average functions defined')
print()
print('Methods to test:')
print('  1. SMA-3  : Simple 3-month Moving Average')
print('  2. SMA-6  : Simple 6-month Moving Average')
print('  3. SMA-12 : Simple 12-month Moving Average')
print('  4. WMA-6  : Weighted 6-month Moving Average')
print('  5. EXP    : Exponential Smoothing (alpha=0.3)')

### Test Set Evaluation (Oct–Dec 2025)

First evaluate on the same Oct–Dec 2025 test set used for Prophet, LSTM, and ARIMA — to see how MA performs historically.

In [ ]:
TEST_MONTHS = 3
train = monthly.iloc[:-TEST_MONTHS].copy()
test  = monthly.iloc[-TEST_MONTHS:].copy()

methods = {
    'SMA-3' : lambda s: simple_moving_average(s, 3),
    'SMA-6' : lambda s: simple_moving_average(s, 6),
    'SMA-12': lambda s: simple_moving_average(s, 12),
    'WMA-6' : lambda s: weighted_moving_average(s, 6),
    'EXP'   : lambda s: exponential_smoothing(s, alpha=0.3)
}

test_results = {}

print('=== TEST SET EVALUATION (Oct-Dec 2025) ===')
print()

for method_name, method_fn in methods.items():
    qty_preds, rev_preds = [], []
    qty_series = train['total_qty'].copy()
    rev_series = train['total_revenue'].copy()

    # Rolling one-step-ahead forecast for each test month
    for i in range(TEST_MONTHS):
        qty_pred = method_fn(qty_series)
        rev_pred = method_fn(rev_series)
        qty_preds.append(qty_pred)
        rev_preds.append(rev_pred)

        # Add actual test value to series for next prediction (rolling)
        qty_series = pd.concat([qty_series,
            pd.Series([test['total_qty'].iloc[i]])], ignore_index=True)
        rev_series = pd.concat([rev_series,
            pd.Series([test['total_revenue'].iloc[i]])], ignore_index=True)

    qty_actual = test['total_qty'].values
    rev_actual = test['total_revenue'].values

    qty_mape = np.mean([compute_mape(a, p) for a, p in zip(qty_actual, qty_preds)])
    rev_mape = np.mean([compute_mape(a, p) for a, p in zip(rev_actual, rev_preds)])
    avg_mape = (qty_mape + rev_mape) / 2

    test_results[method_name] = {
        'qty_preds' : qty_preds,
        'rev_preds' : rev_preds,
        'qty_mape'  : qty_mape,
        'rev_mape'  : rev_mape,
        'avg_mape'  : avg_mape
    }

    print(f'{method_name:8s} → Qty MAPE: {qty_mape:.2f}%  Rev MAPE: {rev_mape:.2f}%  Avg: {avg_mape:.2f}%')

best_test_method = min(test_results, key=lambda x: test_results[x]['avg_mape'])
print()
print(f'Best MA method on test set : {best_test_method} ({test_results[best_test_method]["avg_mape"]:.2f}% avg MAPE)')
print()
print('--- Prophet test set benchmark ---')
print(f'Prophet → Qty MAPE: 19.08%  Rev MAPE: 11.92%  Avg: 15.50%')

### Forecast January 2026 Using All MA Methods

Retrain on full 36 months (Jan 2023 – Dec 2025) then forecast January 2026.

In [ ]:
full_qty_series = monthly['total_qty'].copy()
full_rev_series = monthly['total_revenue'].copy()

jan26_forecasts = {}

for method_name, method_fn in methods.items():
    qty_fc = method_fn(full_qty_series)
    rev_fc = method_fn(full_rev_series)
    jan26_forecasts[method_name] = {
        'qty': qty_fc,
        'rev': rev_fc
    }

print('=== JANUARY 2026 FORECASTS — ALL MA METHODS ===')
print(f'{"Method":<10} {"Pred Qty":>15} {"Pred Rev (B)":>15}')
print('-' * 45)
for method, vals in jan26_forecasts.items():
    print(f'{method:<10} {vals["qty"]:>15,.0f} {vals["rev"]/1e9:>14.2f}B')
print()
print(f'{"ACTUAL":<10} {actual_jan26_qty:>15,.0f} {actual_jan26_rev/1e9:>14.2f}B')

### Validate All MA Methods Against January 2026 Actuals

In [ ]:
jan26_results = {}

print('=== JANUARY 2026 VALIDATION — MA METHODS vs ACTUAL ===')
print(f'Actual Qty : {actual_jan26_qty:,.0f} units')
print(f'Actual Rev : LAK {actual_jan26_rev/1e9:.2f}B')
print()
print(f'{"Method":<10} {"Pred Qty":>12} {"Qty Err%":>10} {"Pred Rev(B)":>13} {"Rev Err%":>10} {"Avg MAPE":>10}')
print('-' * 70)

for method, vals in jan26_forecasts.items():
    qty_mape = compute_mape(actual_jan26_qty, vals['qty'])
    rev_mape = compute_mape(actual_jan26_rev, vals['rev'])
    qty_err  = (vals['qty'] - actual_jan26_qty) / actual_jan26_qty * 100
    rev_err  = (vals['rev'] - actual_jan26_rev) / actual_jan26_rev * 100
    avg_mape = (qty_mape + rev_mape) / 2

    jan26_results[method] = {
        'qty_pred' : vals['qty'],
        'rev_pred' : vals['rev'],
        'qty_mape' : qty_mape,
        'rev_mape' : rev_mape,
        'avg_mape' : avg_mape,
        'qty_err'  : qty_err,
        'rev_err'  : rev_err
    }

    print(f'{method:<10} {vals["qty"]:>12,.0f} {qty_err:>+9.1f}% {vals["rev"]/1e9:>12.2f}B {rev_err:>+9.1f}% {avg_mape:>9.2f}%')

best_ma = min(jan26_results, key=lambda x: jan26_results[x]['avg_mape'])
print()
print(f'Best MA method (Jan 2026) : {best_ma} ({jan26_results[best_ma]["avg_mape"]:.2f}% avg MAPE)')
print()
print('--- Prophet Jan 2026 benchmark ---')
print(f'Prophet → Qty MAPE: 7.34%  Rev MAPE: 10.58%  Avg: 8.96%')

### Final Head-to-Head: Best MA vs Prophet vs All Models

In [ ]:
# All models comparison on Jan 2026 actuals
all_models = {
    'Prophet'        : {'qty_mape': 7.34,  'rev_mape': 10.58, 'avg_mape': 8.96,
                        'qty_pred': 843744, 'rev_pred': 240256227466},
    'LSTM'           : {'qty_mape': 45.02, 'rev_mape': 48.31, 'avg_mape': 46.66,
                        'qty_pred': 500682, 'rev_pred': 138893950976},
    'ARIMA'          : {'qty_mape': 26.25, 'rev_mape': 26.20, 'avg_mape': 26.23,
                        'qty_pred': 671544, 'rev_pred': 198275932379},
    'Ensemble'       : {'qty_mape': 26.21, 'rev_mape': 28.36, 'avg_mape': 27.28,
                        'qty_pred': 671990, 'rev_pred': 192480000000},
}

# Add best MA method
best_ma_result = jan26_results[best_ma]
all_models[f'{best_ma} (JDE Baseline)'] = {
    'qty_mape': best_ma_result['qty_mape'],
    'rev_mape': best_ma_result['rev_mape'],
    'avg_mape': best_ma_result['avg_mape'],
    'qty_pred': best_ma_result['qty_pred'],
    'rev_pred': best_ma_result['rev_pred']
}

# Add all MA methods
for method, result in jan26_results.items():
    if method != best_ma:
        all_models[f'{method}'] = {
            'qty_mape': result['qty_mape'],
            'rev_mape': result['rev_mape'],
            'avg_mape': result['avg_mape'],
            'qty_pred': result['qty_pred'],
            'rev_pred': result['rev_pred']
        }

print('=' * 75)
print('   COMPLETE MODEL COMPARISON — JANUARY 2026 ACTUAL VALIDATION')
print('=' * 75)
print(f'Actual Qty : {actual_jan26_qty:,.0f} units')
print(f'Actual Rev : LAK {actual_jan26_rev/1e9:.2f}B')
print()
print(f'{"Model":<25} {"Qty MAPE":>10} {"Rev MAPE":>10} {"Avg MAPE":>10}')
print('-' * 60)

sorted_models = sorted(all_models.items(), key=lambda x: x[1]['avg_mape'])
for i, (model, metrics) in enumerate(sorted_models):
    marker = ' ← WINNER' if i == 0 else (' ← JDE Baseline' if 'JDE' in model else '')
    print(f'{model:<25} {metrics["qty_mape"]:>9.2f}% {metrics["rev_mape"]:>9.2f}% {metrics["avg_mape"]:>9.2f}%{marker}')

print()
# Prophet vs best MA improvement
prophet_avg = 8.96
best_ma_avg = best_ma_result['avg_mape']
improvement = ((best_ma_avg - prophet_avg) / best_ma_avg) * 100
print(f'Prophet vs {best_ma} (JDE Baseline):')
print(f'  Prophet avg MAPE  : {prophet_avg:.2f}%')
print(f'  {best_ma} avg MAPE : {best_ma_avg:.2f}%')
print(f'  Improvement       : {improvement:.1f}% reduction in forecast error')

### Visualization: All Models vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('January 2026 Forecast vs Actual — All Models + Moving Average Baselines',
             fontsize=13, fontweight='bold')

# Models to plot
plot_models = {
    'Prophet'   : {'qty': 843744,          'rev': 240256227466,   'color': '#3B82F6'},
    'LSTM'      : {'qty': 500682,          'rev': 138893950976,   'color': '#F59E0B'},
    'ARIMA'     : {'qty': 671544,          'rev': 198275932379,   'color': '#10B981'},
    'SMA-3'     : {'qty': jan26_forecasts['SMA-3']['qty'],  'rev': jan26_forecasts['SMA-3']['rev'],  'color': '#EF4444'},
    'SMA-6'     : {'qty': jan26_forecasts['SMA-6']['qty'],  'rev': jan26_forecasts['SMA-6']['rev'],  'color': '#EC4899'},
    'SMA-12'    : {'qty': jan26_forecasts['SMA-12']['qty'], 'rev': jan26_forecasts['SMA-12']['rev'], 'color': '#8B5CF6'},
    'WMA-6'     : {'qty': jan26_forecasts['WMA-6']['qty'],  'rev': jan26_forecasts['WMA-6']['rev'],  'color': '#F97316'},
    'EXP'       : {'qty': jan26_forecasts['EXP']['qty'],    'rev': jan26_forecasts['EXP']['rev'],    'color': '#6B7280'},
}

model_names = list(plot_models.keys())
qty_preds   = [v['qty'] for v in plot_models.values()]
rev_preds   = [v['rev'] for v in plot_models.values()]
bar_colors  = [v['color'] for v in plot_models.values()]

x = np.arange(len(model_names))

# Quantity chart
bars1 = axes[0].bar(x, qty_preds, color=bar_colors, alpha=0.85, width=0.6)
axes[0].axhline(actual_jan26_qty, color='black', linewidth=2.5,
                linestyle='--', label=f'Actual: {actual_jan26_qty:,.0f}')
axes[0].set_title('Quantity Forecast vs Actual (Jan 2026)', fontweight='bold')
axes[0].set_ylabel('Quantity (Units)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
axes[0].legend()
for bar, model in zip(bars1, model_names):
    mape = all_models.get(model, jan26_results.get(model, {})).get('qty_mape',
           compute_mape(actual_jan26_qty, bar.get_height()))
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                 f'{mape:.1f}%', ha='center', fontsize=7, fontweight='bold')

# Revenue chart
bars2 = axes[1].bar(x, [r/1e9 for r in rev_preds], color=bar_colors, alpha=0.85, width=0.6)
axes[1].axhline(actual_jan26_rev/1e9, color='black', linewidth=2.5,
                linestyle='--', label=f'Actual: LAK {actual_jan26_rev/1e9:.2f}B')
axes[1].set_title('Revenue Forecast vs Actual (Jan 2026)', fontweight='bold')
axes[1].set_ylabel('Revenue (LAK Billions)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}B'))
axes[1].legend()
for bar, model in zip(bars2, model_names):
    rev_val = bar.get_height() * 1e9
    mape = compute_mape(actual_jan26_rev, rev_val)
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                 f'{mape:.1f}%', ha='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/15_ma_vs_prophet_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/15_ma_vs_prophet_comparison.png')

### Historical Trend: MA Predictions vs Actuals

In [ ]:
# Plot historical series with MA predictions overlaid
fig, ax = plt.subplots(figsize=(16, 6))

# Historical actual
ax.plot(monthly['ds'], monthly['total_qty'],
        color='#CBD5E1', linewidth=2, marker='o', markersize=3,
        label='Actual (Historical)', zorder=3)

# January 2026 actual
jan26_date = pd.Timestamp('2026-01-01')
ax.scatter(jan26_date, actual_jan26_qty, s=150, color='black',
           zorder=6, label=f'Actual Jan 2026: {actual_jan26_qty:,.0f}', marker='*')

# Plot each MA forecast
ma_colors = {
    'SMA-3' : '#EF4444',
    'SMA-6' : '#EC4899',
    'SMA-12': '#8B5CF6',
    'WMA-6' : '#F97316',
    'EXP'   : '#6B7280'
}
last_actual_date = monthly['ds'].max()
last_actual_qty  = monthly['total_qty'].iloc[-1]

for method, color in ma_colors.items():
    fc = jan26_forecasts[method]['qty']
    mape = jan26_results[method]['qty_mape']
    ax.plot([last_actual_date, jan26_date], [last_actual_qty, fc],
            linestyle='--', color=color, linewidth=1.5, alpha=0.8)
    ax.scatter(jan26_date, fc, s=80, color=color, zorder=5,
               label=f'{method}: {fc:,.0f} (MAPE {mape:.1f}%)')

# Prophet forecast
ax.plot([last_actual_date, jan26_date], [last_actual_qty, 843744],
        linestyle='-', color='#3B82F6', linewidth=2.5)
ax.scatter(jan26_date, 843744, s=120, color='#3B82F6', zorder=6,
           marker='D', label='Prophet: 843,744 (MAPE 7.3%)')

ax.axvline(jan26_date, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.text(jan26_date, monthly['total_qty'].max()*0.95,
        ' Jan 2026', color='gray', fontsize=9)

ax.set_title('Monthly Quantity — Historical + All Forecasts for January 2026',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Quantity (Units)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.legend(loc='upper left', fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('reports/16_historical_all_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/16_historical_all_forecasts.png')

### Final Summary & Interview Narrative

In [ ]:
best_ma_avg  = jan26_results[best_ma]['avg_mape']
prophet_avg  = 8.96
improvement  = ((best_ma_avg - prophet_avg) / best_ma_avg) * 100

print('=' * 65)
print('   FINAL SUMMARY — MOVING AVERAGE vs PROPHET')
print('=' * 65)
print()
print('ACTUAL JANUARY 2026:')
print(f'  Quantity : {actual_jan26_qty:,.0f} units')
print(f'  Revenue  : LAK {actual_jan26_rev/1e9:.2f} Billion')
print()
print('MOVING AVERAGE RESULTS (JDE Baseline):')
for method, result in sorted(jan26_results.items(), key=lambda x: x[1]['avg_mape']):
    print(f'  {method:<8}: Qty MAPE={result["qty_mape"]:.2f}%  Rev MAPE={result["rev_mape"]:.2f}%  Avg={result["avg_mape"]:.2f}%')
print()
print('PROPHET (ML Model):')
print(f'  Qty MAPE : 7.34%')
print(f'  Rev MAPE : 10.58%')
print(f'  Avg MAPE : {prophet_avg:.2f}%')
print()
print('IMPROVEMENT:')
print(f'  Best MA ({best_ma}) avg MAPE : {best_ma_avg:.2f}%')
print(f'  Prophet avg MAPE           : {prophet_avg:.2f}%')
print(f'  Error reduction            : {improvement:.1f}%')
print()
print('INTERVIEW NARRATIVE:')
print(f'  "We benchmarked Prophet against JDE\'s native Moving Average')
print(f'  forecasting on the exact same January 2026 holdout validation set.')
print(f'  The best Moving Average method ({best_ma}) achieved {best_ma_avg:.1f}% average MAPE,')
print(f'  while Prophet achieved {prophet_avg}% — a {improvement:.0f}% reduction in forecast')
print(f'  error. This was validated on real client data, not simulated results."')